# Games Backtest (2025 default)

Backtest variant of `games_today_tomorrow.ipynb`:

- Defaults to **completed 2025 games** from `data/games_2025.parquet`.
- Keeps V6-style scoring logic, but changes data selection upstream.
- If Kalshi historical implied columns are missing, scoring still runs and Kalshi-based metrics are omitted (set to NaN).

Run top-to-bottom after refreshing Parquet files.

In [9]:
# Backtest parameters (edit these)
SEASON = 2025
START_DATE = "2025-03-27"  # inclusive, YYYY-MM-DD
END_DATE = "2025-11-11"             # inclusive; None = today

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

print({
    "SEASON": SEASON,
    "START_DATE": START_DATE,
    "END_DATE": END_DATE or "today",
    "FINAL_STATES": sorted(FINAL_STATES),
})

{'SEASON': 2025, 'START_DATE': '2025-03-27', 'END_DATE': '2025-11-11', 'FINAL_STATES': ['Completed Early', 'Final', 'Game Over']}


In [10]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run: python -m app.main live --season {SEASON}")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

df_bt = df[df["detailed_state"].isin(FINAL_STATES)].copy()

if "home_win" in df_bt.columns:
    # ensure final training label is present for backtest
    df_bt = df_bt[df_bt["home_win"].notna()].copy()

start_ts = pd.Timestamp(START_DATE).normalize()
end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE).normalize()
if end_ts < start_ts:
    raise ValueError(f"END_DATE ({end_ts.date()}) must be >= START_DATE ({start_ts.date()})")

sub = df_bt[df_bt["game_date"].between(start_ts, end_ts)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)}  completed={len(df_bt)}")
if len(df_bt):
    print(f"Date range completed: {df_bt['game_date'].min().date()} .. {df_bt['game_date'].max().date()}")
print(f"Backtest window: {start_ts.date()} .. {end_ts.date()}  rows={len(sub)}")


Using: C:\Users\Austin\baseball\mlb-model\data\games_2025.parquet
Rows total=3013  completed=2894
Date range completed: 2025-03-01 .. 2025-11-01
Backtest window: 2025-03-27 .. 2025-11-11  rows=2581


In [11]:
# Core thresholds from your mismatch workflow
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2

USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

favorites = s.copy().sort_values(["game_date", "home_team_name"]).reset_index(drop=True)
favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = np.where(
    favorites["_mismatch_side"].eq("home_favored"),
    favorites["home_team_name"],
    favorites["away_team_name"],
)

print("Favorites frame rows:", len(favorites))
display(favorites.tail(25))

Favorites frame rows: 2581


,game_pk,game_date,detailed_state,home_team_name,away_team_name,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher,away_probable_pitcher,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_kbb_diff,sp_xfip_diff,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_fip,away_pen_season_fip,home_pen_season_era,away_pen_season_era,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,_mismatch_side,core_matches,favored_team
2556,813055,2025-10-08,Final,Detroit Tigers,Seattle Mariners,116,136,9.0,3.0,Casey Mize,Bryce Miller,663554,682243,R,R,101.461378,102.992345,16.427432,10.204082,3.851678,5.136900,1.0,6.223351,-1.285223,-1.530967,98.233996,103.062914,-4.828918,33.333333,0.000000,0.573684,10.100000,4.075678,3.777122,3.583739,3.636531,1.651724,2.860000,1.396552,4.320000,-2.423953,-0.917122,NaN,NaN,NaN,2025,0.0,0.0,False,False,False,False,home_favored,0.0,Detroit Tigers
2557,813041,2025-10-08,Final,Los Angeles Dodgers,Philadelphia Phillies,119,143,2.0,8.0,Yoshinobu Yamamoto,Aaron Nola,808967,605400,R,R,106.889353,105.636743,20.760234,17.079208,2.904223,4.541696,0.0,3.681026,-1.637473,1.252610,106.236203,105.408389,0.827815,20.833333,34.615385,1.766667,2.475000,3.881964,4.100000,4.002471,4.142857,1.864706,2.561538,2.117647,2.076923,-2.017258,-1.538462,NaN,NaN,NaN,2025,2.0,0.0,True,False,False,False,home_favored,2.0,Los Angeles Dodgers
2558,813060,2025-10-08,Final,New York Yankees,Toronto Blue Jays,147,141,2.0,5.0,Cam Schlittler,Louis Varland,693645,686973,R,R,109.533751,105.775922,17.434211,17.905405,3.702740,3.100000,0.0,-0.471195,0.602740,3.757829,108.029801,105.270419,2.759382,30.769231,36.363636,1.814286,0.766667,3.758587,4.008023,3.982687,3.980431,2.933333,3.100000,0.000000,1.000000,-0.825254,-0.908023,NaN,NaN,NaN,2025,0.0,2.0,False,True,False,True,away_favored,2.0,Toronto Blue Jays
2559,813050,2025-10-09,Final,Chicago Cubs,Milwaukee Brewers,112,158,6.0,0.0,Matthew Boyd,Freddy Peralta,571510,642547,L,R,104.384134,102.296451,15.598886,19.087137,3.612059,3.609434,1.0,-3.488251,0.002625,2.087683,103.614790,104.365136,-0.750346,NaN,33.333333,NaN,6.600000,3.967769,3.396875,3.849174,2.973214,1.728571,2.700000,2.314286,0.900000,-2.239197,-0.696875,NaN,NaN,NaN,2025,1.0,0.0,False,False,False,False,home_favored,1.0,Chicago Cubs
2560,813042,2025-10-09,Final,Los Angeles Dodgers,Philadelphia Phillies,119,143,2.0,1.0,Tyler Glasnow,Cristopher Sánchez,607192,650911,R,L,106.889353,105.636743,17.213115,20.817844,3.719926,2.515842,1.0,-3.604729,1.204085,1.252610,108.772928,105.408389,3.364539,15.384615,44.444444,2.100000,0.276471,3.881964,4.100000,4.002471,4.142857,1.242857,2.463636,0.642857,2.454545,-2.639107,-1.636364,NaN,NaN,NaN,2025,0.0,1.0,False,False,False,False,away_favored,1.0,Philadelphia Phillies
2561,813054,2025-10-10,Final,Seattle Mariners,Detroit Tigers,136,116,3.0,2.0,George Kirby,Tarik Skubal,669923,669373,R,L,102.992345,101.461378,20.610687,27.807487,3.330159,2.413993,1.0,-7.196800,0.916166,1.530967,103.369828,98.233996,5.135832,45.000000,NaN,2.900000,NaN,3.777122,4.075678,3.636531,3.583739,2.900000,1.745161,4.500000,1.741935,-0.877122,-2.330516,NaN,NaN,NaN,2025,0.0,1.0,False,False,False,False,away_favored,1.0,Detroit Tigers
2562,813046,2025-10-11,Final,Milwaukee Brewers,Chicago Cubs,158,112,3.0,1.0,Trevor Megill,Drew Pomeranz,656730,519141,R,L,102.296451,104.384134,22.395833,20.689655,2.461702,3.321477,1.0,1.706178,-0.859774,-2.087683,104.365136,103.614790,0.750346,66.666667,66.666667,-0.900000,-0.900000,3.396875,3.967769,2.9

In [12]:
# V6 model block (kept aligned in spirit with games_today_tomorrow)
s = favorites.copy()

# Stable scaling
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

# Tunable compression / penalty knobs
RISK_W_CONFLICT = 0.30
RISK_W_INSTABILITY = 0.40
RISK_W_SIGNAL_GAP = 0.30
RISK_TANH_ALPHA = 0.80
QUALITY_INSTABILITY_DECAY = 0.90
TRAP_PENALTY_WEIGHT = 0.35

OFF_W_OFFENSE = 0.85
OFF_W_PLATOON = 0.15
PEN_LEG_DAMP = 0.65
COHERENCE_SOFT_MIN = 0.50
COHERENCE_SOFT_RANGE = 0.50

# Missing SP/offense diffs: neutral 0 so norms stay finite
kbb = pd.to_numeric(s["sp_kbb_diff"], errors="coerce").fillna(0.0)
xfi = pd.to_numeric(s["sp_xfip_diff"], errors="coerce").fillna(0.0)
off = pd.to_numeric(s["offense_diff"], errors="coerce").fillna(0.0)
kbb_norm = kbb / SP_KBB_ABS
xfip_norm = -xfi / SP_XFIP_ABS
off_norm = off / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

# Pen: prefer roll14; neutralize missing values so matrix math is stable
if "home_pen_roll14_fip" in s.columns:
    hr_pen = s["home_pen_roll14_fip"]
    if "home_pen_season_fip" in s.columns:
        hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
else:
    hr_pen = pd.Series(np.nan, index=s.index)

if "away_pen_roll14_fip" in s.columns:
    ar_pen = s["away_pen_roll14_fip"]
    if "away_pen_season_fip" in s.columns:
        ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
else:
    ar_pen = pd.Series(np.nan, index=s.index)

pen_gap = hr_pen - ar_pen  # home - away; lower home FIP better
pen_norm = (-pen_gap / PEN_ABS).fillna(0.0)

sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
off_vector = (OFF_W_OFFENSE * off_norm) + (OFF_W_PLATOON * platoon_norm)
pen_vector = PEN_LEG_DAMP * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

agreement = 1 - np.nanvar(sign_matrix, axis=0)
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
coherence_mult = COHERENCE_SOFT_MIN + COHERENCE_SOFT_RANGE * coherence_score
raw_edge = mean_mag
instability = np.std(signal_matrix, axis=0)

direction_conflict = (
    (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
    + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
    + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
)

risk_penalty_raw = (
    RISK_W_CONFLICT * direction_conflict
    + RISK_W_INSTABILITY * instability
    + RISK_W_SIGNAL_GAP * np.abs(sp_vector - off_vector)
)
risk_penalty = np.tanh(RISK_TANH_ALPHA * risk_penalty_raw)

trap_score = raw_edge * (1 - coherence_score)
quality_score = raw_edge * coherence_mult * np.exp(-QUALITY_INSTABILITY_DECAY * instability) * (1 / (1 + risk_penalty))
risk_adjusted_edge = quality_score - TRAP_PENALTY_WEIGHT * trap_score

prefer_home = sp_vector >= 0
s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = direction_conflict
s["instability"] = instability
s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score
s["risk_penalty_raw"] = risk_penalty_raw
s["risk_penalty"] = risk_penalty

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int)
    + (np.abs(xfip_norm) > 1).astype(int)
    + (np.abs(off_norm) > 1).astype(int)
)

# Backtest outcome: did the model's favored side win?
if "home_win" in s.columns:
    hw = pd.to_numeric(s["home_win"], errors="coerce")
    s["favorite_won"] = np.where(prefer_home, hw == 1.0, hw == 0.0)
else:
    s["favorite_won"] = np.nan

# Kalshi is optional in historical backtests.
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(s.columns):
    s["implied_prob"] = np.where(prefer_home, s["kalshi_home_implied"], s["kalshi_away_implied"])
    s["market_edge"] = s["risk_adjusted_edge"] - s["implied_prob"]
else:
    s["implied_prob"] = np.nan
    s["market_edge"] = np.nan

scored = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False]).reset_index(drop=True)

show_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side", "favorite_won",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "signal_agreement", "signal_conflict", "core_matches",
    "sp_kbb_diff", "sp_xfip_diff", "offense_diff", "offense_platoon_diff",
    "implied_prob", "market_edge", "home_win",
]
show_cols = [c for c in show_cols if c in scored.columns]

print(f"Scored games: {len(scored)}")
display(scored[show_cols].head(25))

# Optional aggregate view (uncomment to use):
# display(
#     scored["favorite_won"]
#     .map({True: "Favorite Won", False: "Favorite Did Not Win"})
#     .fillna("Unknown")
#     .value_counts(dropna=False)
#     .rename_axis("favorite_result")
#     .to_frame("count")
# )

Scored games: 2581


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,favorite_won,risk_adjusted_edge,quality_score,coherence_score,instability,signal_agreement,signal_conflict,core_matches,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,implied_prob,market_edge,home_win
0,2025-04-06,Pittsburgh Pirates,New York Yankees,New York Yankees,away_favored,False,0.805786,1.045496,0.683944,0.203132,0,3,3,-6.326490,1.522131,-18.371608,-21.987365,NaN,NaN,1.0
1,2025-08-28,Chicago White Sox,New York Yankees,New York Yankees,away_favored,True,0.716448,0.875649,0.696088,0.136699,0,3,3,-5.590776,0.569815,-15.588031,-16.142384,NaN,NaN,0.0
2,2025-05-18,Chicago Cubs,Chicago White Sox,Chicago Cubs,home_favored,True,0.636993,0.753484,0.713128,0.054964,3,0,2,2.237335,-1.017705,10.438413,11.727373,NaN,NaN,1.0
3,2025-05-14,New York Mets,Pittsburgh Pirates,New York Mets,home_favored,False,0.512360,0.650768,0.684137,0.202016,3,0,2,2.141548,-0.831361,13.639527,7.626495,NaN,NaN,0.0
4,2025-05-02,New York Yankees,Tampa Bay Rays,New York Yankees,home_favored,True,0.489471,0.605584,0.693512,0.150169,3,0,2,1.617271,-1.289217,10.160056,13.191183,NaN,NaN,1.0
5,2025-06-12,Chicago Cubs,Pittsburgh Pirates,Chicago Cubs,home_favored,True,0.475679,0.637623,0.673132,0.269166,3,0,3,5.098065,-0.933036,13.221990,15.446771,NaN,NaN,1.0
6,2025-09-02,St. Louis Cardinals,Athletics,Athletics,away_favored,False,0.474094,0.556406,0.716891,0.038470,0,3,1,-1.079414,0.928475,-7.794015,-8.278146,NaN,NaN,1.0
7,2025-08-27,New York Yankees,Washington Nationals,New York Yankees,home_favored,True,0.464014,0.682575,0.661507,0.348770,3,0,3,5.812939,-1.463813,13.082811,17.172415,NaN,NaN,1.0
8,2025-08-29,Chicago White Sox,New York Yankees,New York Yankees,away_favored,True,0.455965,0.663878,0.662945,0.338387,0,3,3,-5.285217,1.477289,-15.588031,-8.641205,NaN,NaN,0.0
9,2025-06-29,Texas Rangers,Seattle Mariners,Seattle Mariners,away_favored,True,0.447928,0.526404,0.715859,0.042943,0,3,1,-3.031770,0.273687,-7.933194,-7.174393,NaN,NaN,0.0


In [13]:
# Additional breakdowns by metric bands
band_df = scored.copy()

# Keep known outcomes for cleaner win-rate summaries
band_df = band_df[band_df["favorite_won"].isin([True, False])].copy()

band_specs = {
    "risk_adjusted_edge": [-np.inf, 0.25, 0.50, 0.75, 1.00, np.inf],
    "quality_score": [-np.inf, 0.10, 0.20, 0.30, 0.40, np.inf],
    "coherence_score": [-np.inf, 0.40, 0.50, 0.60, 0.70, np.inf],
    "instability": [-np.inf, 0.80, 1.00, 1.20, 1.50, np.inf],
}

for metric, bins in band_specs.items():
    if metric not in band_df.columns:
        continue

    d = band_df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        continue

    d["band"] = pd.cut(d[metric], bins=bins, include_lowest=True)

    out = (
        d.groupby("band", observed=False)["favorite_won"]
        .agg(
            games="count",
            favorite_wins=lambda x: int((x == True).sum()),
            favorite_losses=lambda x: int((x == False).sum()),
            favorite_win_rate="mean",
        )
        .reset_index()
    )
    out["favorite_win_rate"] = out["favorite_win_rate"].round(3)

    print(f"\n{metric} band breakdown")
    display(out)



risk_adjusted_edge band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.25]",2500,1404,1096,0.562
1,"(0.25, 0.5]",77,48,29,0.623
2,"(0.5, 0.75]",3,2,1,0.667
3,"(0.75, 1.0]",1,0,1,0.000
4,"(1.0, inf]",0,0,0,NaN



quality_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.1]",1648,883,765,0.536
1,"(0.1, 0.2]",456,278,178,0.610
2,"(0.2, 0.3]",297,173,124,0.582
3,"(0.3, 0.4]",108,74,34,0.685
4,"(0.4, inf]",72,46,26,0.639



coherence_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.4]",1163,641,522,0.551
1,"(0.4, 0.5]",605,313,292,0.517
2,"(0.5, 0.6]",336,220,116,0.655
3,"(0.6, 0.7]",468,276,192,0.590
4,"(0.7, inf]",9,4,5,0.444



instability band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.8]",753,410,343,0.544
1,"(0.8, 1.0]",304,171,133,0.562
2,"(1.0, 1.2]",333,191,142,0.574
3,"(1.2, 1.5]",369,210,159,0.569
4,"(1.5, inf]",822,472,350,0.574


In [14]:
# Decision / confidence layers + backtest summary
def decision_layer(df: pd.DataFrame) -> pd.Series:
    # Relaxed vs 1.0/0.60/1.2 and 0.55/0.45 — avoids starving BET count
    conditions = [
        (df["risk_adjusted_edge"] > 0.72) & (df["coherence_score"] > 0.52) & (df["instability"] < 1.35),
        (df["risk_adjusted_edge"] > 0.48) & (df["coherence_score"] > 0.40),
    ]
    choices = ["BET", "LEAN"]
    return np.select(conditions, choices, default="PASS")


def confidence_tier(df: pd.DataFrame) -> pd.Series:
    return np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.68),
        "A (Strong Bet)",
        np.where(
            df["decision"] == "BET",
            "B (Playable Bet)",
            np.where(df["decision"] == "LEAN", "C (Lean Only)", "D (Pass)"),
        ),
    )


bt = scored.copy()
bt["decision"] = decision_layer(bt)
bt["tier"] = confidence_tier(bt)

# Evaluate pick correctness only on non-PASS rows
bt_eval = bt[bt["decision"].isin(["BET", "LEAN"])].copy()
bt_eval["pick_home"] = bt_eval["_mismatch_side"].eq("home_favored")
bt_eval["won"] = np.where(bt_eval["pick_home"], bt_eval["home_win"] == 1.0, bt_eval["home_win"] == 0.0)

summary = {
    "rows_scored": int(len(bt)),
    "rows_bet_or_lean": int(len(bt_eval)),
    "win_rate_bet_or_lean": float(bt_eval["won"].mean()) if len(bt_eval) else np.nan,
    "bets_only": int((bt["decision"] == "BET").sum()),
    "leans_only": int((bt["decision"] == "LEAN").sum()),
}

print(summary)
if len(bt_eval):
    by_tier = bt_eval.groupby("tier")["won"].agg(["count", "mean"]).rename(columns={"mean": "win_rate"})
    display(by_tier.sort_values("count", ascending=False))

view_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "decision", "tier",
    "risk_adjusted_edge", "coherence_score", "instability", "home_win", "won",
]
view_cols = [c for c in view_cols if c in bt_eval.columns]
display(bt_eval.sort_values("risk_adjusted_edge", ascending=False)[view_cols].head(50))

# Slate Top-N evaluation (default Top-3 per date) on ALL scored games with known outcomes
TOP_N = 3
if "favorite_won" in scored.columns:
    top_pool = scored[scored["favorite_won"].isin([True, False])].copy()
    top_pool["won"] = top_pool["favorite_won"].astype(bool)
    if len(top_pool):
        bt_topn = (
            top_pool.sort_values(["game_date", "risk_adjusted_edge"], ascending=[True, False])
            .groupby("game_date", as_index=False, group_keys=False)
            .head(TOP_N)
            .copy()
        )
        topn_summary = {
            "top_n": TOP_N,
            "slates": int(bt_topn["game_date"].nunique()),
            "topn_rows": int(len(bt_topn)),
            "topn_win_rate": float(bt_topn["won"].mean()) if len(bt_topn) else np.nan,
        }
        print("Top-N slate summary:", topn_summary)
        by_date = bt_topn.groupby("game_date")["won"].agg(picks="count", wins="sum", win_rate="mean").reset_index()
        display(by_date.head(20))
        top_cols = [c for c in view_cols if c in bt_topn.columns] + [c for c in ["favorite_won"] if c in bt_topn.columns]
        display(bt_topn.sort_values(["game_date", "risk_adjusted_edge"], ascending=[False, False])[top_cols].head(50))

{'rows_scored': 2581, 'rows_bet_or_lean': 3, 'win_rate_bet_or_lean': 0.6666666666666666, 'bets_only': 0, 'leans_only': 3}


,count,win_rate
tier,,
C (Lean Only),3,0.666667


,game_date,home_team_name,away_team_name,favored_team,decision,tier,risk_adjusted_edge,coherence_score,instability,home_win,won
0,2025-04-06,Pittsburgh Pirates,New York Yankees,New York Yankees,LEAN,C (Lean Only),0.805786,0.683944,0.203132,1.0,False
1,2025-08-28,Chicago White Sox,New York Yankees,New York Yankees,LEAN,C (Lean Only),0.716448,0.696088,0.136699,0.0,True
2,2025-05-18,Chicago Cubs,Chicago White Sox,Chicago Cubs,LEAN,C (Lean Only),0.636993,0.713128,0.054964,1.0,True


Top-N slate summary: {'top_n': 3, 'slates': 208, 'topn_rows': 589, 'topn_win_rate': 0.5636672325976231}


,game_date,picks,wins,win_rate
0,2025-03-27,3,1,0.333333
1,2025-03-28,3,3,1.000000
2,2025-03-29,3,1,0.333333
3,2025-03-30,3,2,0.666667
4,2025-03-31,3,2,0.666667
5,2025-04-01,3,2,0.666667
6,2025-04-02,3,1,0.333333
7,2025-04-03,3,0,0.000000
8,2025-04-04,3,3,1.000000
9,2025-04-05,3,1,0.333333


,game_date,home_team_name,away_team_name,favored_team,risk_adjusted_edge,coherence_score,instability,home_win,won,favorite_won
1649,2025-11-01,Toronto Blue Jays,Los Angeles Dodgers,Los Angeles Dodgers,-0.189866,0.555466,2.152061,0.0,True,True
314,2025-10-31,Toronto Blue Jays,Los Angeles Dodgers,Los Angeles Dodgers,0.095645,0.657123,0.381445,0.0,True,True
326,2025-10-29,Los Angeles Dodgers,Toronto Blue Jays,Los Angeles Dodgers,0.091740,0.682908,0.209158,0.0,False,False
1286,2025-10-28,Los Angeles Dodgers,Toronto Blue Jays,Los Angeles Dodgers,-0.121020,0.567505,1.736659,0.0,False,False
324,2025-10-27,Los Angeles Dodgers,Toronto Blue Jays,Los Angeles Dodgers,0.091848,0.652262,0.419576,1.0,True,True
313,2025-10-25,Toronto Blue Jays,Los Angeles Dodgers,Los Angeles Dodgers,0.095645,0.657123,0.381445,0.0,True,True
325,2025-10-24,Toronto Blue Jays,Los Angeles Dodgers,Los Angeles Dodgers,0.091740,0.682908,0.209158,1.0,False,False
763,2025-10-20,Toronto Blue Jays,Seattle Mariners,Seattle Mariners,-0.029917,0.434256,0.581245,1.0,False,False
1121,2025-10-19,Toronto Blue Jays,Seattle Mariners,Seattle Mariners,-0.092389,0.406540,0.917359,1.0,False,False
1627,2025-10-17,Seattle Mariners,Toronto Blue Jays,Toronto Blue Jays,-0.185946,0.380323,1.399916,1.0,False,False


In [15]:
# Quantile calibration tables + monotonicity checks

cal_df = scored.copy()
if "favorite_won" not in cal_df.columns:
    raise ValueError("favorite_won missing; run V6 scoring cell first.")

cal_df = cal_df[cal_df["favorite_won"].isin([True, False])].copy()
if cal_df.empty:
    raise ValueError("No rows with known favorite_won values for calibration.")

metrics = [
    "risk_adjusted_edge",
    "quality_score",
    "coherence_score",
    "instability",
]


def quantile_calibration_table(df: pd.DataFrame, metric: str, q: int = 10, ascending_good: bool = True):
    d = df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        return None, None

    # Handle low-variance metrics safely.
    n_unique = d[metric].nunique(dropna=True)
    q_eff = max(2, min(q, int(n_unique)))

    d["quantile"] = pd.qcut(d[metric], q=q_eff, duplicates="drop")
    tab = (
        d.groupby("quantile", observed=False)["favorite_won"]
        .agg(games="count", favorite_wins="sum", favorite_win_rate="mean")
        .reset_index()
    )
    tab["favorite_losses"] = tab["games"] - tab["favorite_wins"]
    tab["favorite_win_rate"] = tab["favorite_win_rate"].round(3)

    rates = tab["favorite_win_rate"].to_numpy()
    if not ascending_good:
        rates = rates[::-1]
    deltas = np.diff(rates)
    mono_score = float((deltas >= 0).mean()) if len(deltas) else np.nan
    is_monotonic = bool((deltas >= 0).all()) if len(deltas) else True

    summary = {
        "metric": metric,
        "quantile_bins": int(len(tab)),
        "rows_used": int(tab["games"].sum()),
        "win_rate_first_bin": float(tab.iloc[0]["favorite_win_rate"]),
        "win_rate_last_bin": float(tab.iloc[-1]["favorite_win_rate"]),
        "lift_last_minus_first": float(tab.iloc[-1]["favorite_win_rate"] - tab.iloc[0]["favorite_win_rate"]),
        "is_monotonic_expected_direction": is_monotonic,
        "monotonicity_fraction": round(mono_score, 3) if not np.isnan(mono_score) else np.nan,
    }
    return tab, summary


# For instability, lower values are generally better; reverse expected direction.
expected_ascending = {
    "risk_adjusted_edge": True,
    "quality_score": True,
    "coherence_score": True,
    "instability": False,
}

summaries = []
for m in metrics:
    if m not in cal_df.columns:
        continue
    tab, s = quantile_calibration_table(cal_df, m, q=10, ascending_good=expected_ascending[m])
    if tab is None:
        continue
    print(f"\n{m}: quantile calibration")
    display(tab)
    summaries.append(s)

if summaries:
    print("\nMonotonicity summary")
    display(pd.DataFrame(summaries).sort_values("metric").reset_index(drop=True))


risk_adjusted_edge: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-1.6869999999999998, -0.41]",259,158,0.610,101
1,"(-0.41, -0.297]",258,139,0.539,119
2,"(-0.297, -0.223]",258,145,0.562,113
3,"(-0.223, -0.17]",258,139,0.539,119
4,"(-0.17, -0.124]",258,148,0.574,110
5,"(-0.124, -0.0762]",258,153,0.593,105
6,"(-0.0762, -0.0312]",258,132,0.512,126
7,"(-0.0312, 0.00961]",258,135,0.523,123
8,"(0.00961, 0.119]",258,150,0.581,108
9,"(0.119, 0.806]",258,155,0.601,103



quality_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-0.0007880000000000001, 0.0464]",259,157,0.606,102
1,"(0.0464, 0.0622]",258,126,0.488,132
2,"(0.0622, 0.0705]",258,134,0.519,124
3,"(0.0705, 0.0768]",258,140,0.543,118
4,"(0.0768, 0.0844]",258,131,0.508,127
5,"(0.0844, 0.0941]",258,141,0.547,117
6,"(0.0941, 0.114]",258,145,0.562,113
7,"(0.114, 0.189]",258,165,0.640,93
8,"(0.189, 0.262]",258,145,0.562,113
9,"(0.262, 1.045]",258,170,0.659,88



coherence_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.256, 0.354]",259,143,0.552,116
1,"(0.354, 0.368]",258,144,0.558,114
2,"(0.368, 0.381]",258,135,0.523,123
3,"(0.381, 0.393]",258,146,0.566,112
4,"(0.393, 0.408]",258,142,0.550,116
5,"(0.408, 0.431]",258,136,0.527,122
6,"(0.431, 0.553]",258,136,0.527,122
7,"(0.553, 0.596]",258,167,0.647,91
8,"(0.596, 0.633]",258,156,0.605,102
9,"(0.633, 0.717]",258,149,0.578,109



instability: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.0375, 0.428]",259,151,0.583,108
1,"(0.428, 0.629]",258,129,0.500,129
2,"(0.629, 0.815]",258,141,0.547,117
3,"(0.815, 0.987]",258,148,0.574,110
4,"(0.987, 1.136]",258,147,0.570,111
5,"(1.136, 1.316]",258,153,0.593,105
6,"(1.316, 1.549]",258,140,0.543,118
7,"(1.549, 1.821]",258,148,0.574,110
8,"(1.821, 2.281]",258,143,0.554,115
9,"(2.281, 9.438]",258,154,0.597,104



Monotonicity summary


,metric,quantile_bins,rows_used,win_rate_first_bin,win_rate_last_bin,lift_last_minus_first,is_monotonic_expected_direction,monotonicity_fraction
0,coherence_score,10,2581,0.552,0.578,0.026,False,0.444
1,instability,10,2581,0.583,0.597,0.014,False,0.444
2,quality_score,10,2581,0.606,0.659,0.053,False,0.667
3,risk_adjusted_edge,10,2581,0.610,0.601,-0.009,False,0.667


In [16]:
# Post-model calibration (logistic + isotonic), quantile diagnostics, and YoY-style summary schema
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

work = scored.copy()
work = work[work["favorite_won"].isin([True, False])].copy()
if work.empty:
    raise ValueError("No rows with favorite_won for calibration.")

work["y"] = work["favorite_won"].astype(int)
work["score_raw"] = pd.to_numeric(work["risk_adjusted_edge"], errors="coerce")
work = work.dropna(subset=["score_raw", "y"]).copy()

# Time-aware split by game_date to reduce leakage.
unique_dates = sorted(pd.to_datetime(work["game_date"]).dropna().unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough unique dates for calibration split.")
cut_idx = max(1, int(len(unique_dates) * 0.7))
train_dates = set(unique_dates[:cut_idx])
valid_dates = set(unique_dates[cut_idx:])

train = work[work["game_date"].isin(train_dates)].copy()
valid = work[work["game_date"].isin(valid_dates)].copy()
if train.empty or valid.empty:
    raise ValueError("Train/valid split empty; adjust date window.")

xtr = train[["score_raw"]].to_numpy()
ytr = train["y"].to_numpy()
xva = valid[["score_raw"]].to_numpy()
yva = valid["y"].to_numpy()

# Baseline probability proxy from score scaling
smin, smax = float(train["score_raw"].min()), float(train["score_raw"].max())
den = (smax - smin) if (smax - smin) > 1e-12 else 1.0
train["p_raw"] = ((train["score_raw"] - smin) / den).clip(0, 1)
valid["p_raw"] = ((valid["score_raw"] - smin) / den).clip(0, 1)

# Logistic (Platt-style)
logit = LogisticRegression(max_iter=2000)
logit.fit(xtr, ytr)
train["p_model_logit"] = logit.predict_proba(xtr)[:, 1]
valid["p_model_logit"] = logit.predict_proba(xva)[:, 1]

# Isotonic on raw score
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(train["score_raw"].to_numpy(), ytr)
train["p_model_iso"] = iso.predict(train["score_raw"].to_numpy())
valid["p_model_iso"] = iso.predict(valid["score_raw"].to_numpy())

# Write calibrated probabilities back to scored
scored = scored.copy()
for col in ["p_model_logit", "p_model_iso"]:
    scored[col] = np.nan
scored.loc[train.index, "p_model_logit"] = train["p_model_logit"]
scored.loc[valid.index, "p_model_logit"] = valid["p_model_logit"]
scored.loc[train.index, "p_model_iso"] = train["p_model_iso"]
scored.loc[valid.index, "p_model_iso"] = valid["p_model_iso"]


def metric_row(name: str, y_true: np.ndarray, p: np.ndarray) -> dict:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        "model": name,
        "brier": float(brier_score_loss(y_true, p)),
        "logloss": float(log_loss(y_true, p)),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else np.nan,
    }

metric_rows = [
    metric_row("raw_scaled", yva, valid["p_raw"].to_numpy()),
    metric_row("logit", yva, valid["p_model_logit"].to_numpy()),
    metric_row("isotonic", yva, valid["p_model_iso"].to_numpy()),
]
metrics_df = pd.DataFrame(metric_rows).sort_values("brier")
print("Validation calibration metrics")
display(metrics_df)


def quantile_hit_rate(df: pd.DataFrame, score_col: str, y_col: str = "y", q: int = 10):
    d = df[[score_col, y_col]].dropna().copy()
    if d.empty or d[score_col].nunique() < 2:
        return pd.DataFrame(), {"metric": score_col, "monotonicity_fraction": np.nan, "lift_last_minus_first": np.nan}
    q_eff = min(q, int(d[score_col].nunique()))
    d["quantile"] = pd.qcut(d[score_col], q=q_eff, duplicates="drop")
    tab = d.groupby("quantile", observed=False)[y_col].agg(games="count", win_rate="mean").reset_index()
    tab["win_rate"] = tab["win_rate"].round(3)
    deltas = np.diff(tab["win_rate"].to_numpy())
    mono = float((deltas >= 0).mean()) if len(deltas) else np.nan
    lift = float(tab.iloc[-1]["win_rate"] - tab.iloc[0]["win_rate"]) if len(tab) else np.nan
    return tab, {"metric": score_col, "monotonicity_fraction": round(mono, 3) if not np.isnan(mono) else np.nan, "lift_last_minus_first": lift}

quant_targets = ["score_raw", "quality_score", "coherence_score", "p_model_logit", "p_model_iso"]
mono_rows = []
for col in quant_targets:
    if col not in valid.columns:
        continue
    qt, ms = quantile_hit_rate(valid, col, y_col="y", q=10)
    if len(qt):
        print(f"\nValidation quantile hit-rate: {col}")
        display(qt)
    mono_rows.append(ms)

mono_df = pd.DataFrame(mono_rows)
print("\nMonotonicity summary")
display(mono_df)

# Standardized summary schema for cross-year comparison
summary_row = {
    "season": int(SEASON),
    "rows_scored": int(len(scored)),
    "rows_with_outcome": int(len(work)),
    "train_rows": int(len(train)),
    "valid_rows": int(len(valid)),
    "top_n_default": 3,
}

# Include top-N result if prior cell produced it
if "bt_topn" in globals() and isinstance(bt_topn, pd.DataFrame) and len(bt_topn):
    summary_row["topn_rows"] = int(len(bt_topn))
    summary_row["topn_win_rate"] = float(bt_topn["won"].mean())
else:
    summary_row["topn_rows"] = np.nan
    summary_row["topn_win_rate"] = np.nan

best_cal = metrics_df.iloc[0]
summary_row["best_model_by_brier"] = best_cal["model"]
summary_row["best_brier"] = float(best_cal["brier"])
summary_row["best_logloss"] = float(best_cal["logloss"])
summary_row["best_auc"] = float(best_cal["auc"]) if pd.notna(best_cal["auc"]) else np.nan

year_summary = pd.DataFrame([summary_row])
print("\nYear summary row")
display(year_summary)

Validation calibration metrics


,model,brier,logloss,auc
1,logit,0.248032,0.689219,0.472845
2,isotonic,0.249381,0.692241,0.500439
0,raw_scaled,0.258344,0.713095,0.527155



Validation quantile hit-rate: score_raw


,quantile,games,win_rate
0,"(-1.0119999999999998, -0.384]",56,0.518
1,"(-0.384, -0.271]",55,0.509
2,"(-0.271, -0.198]",55,0.527
3,"(-0.198, -0.155]",55,0.545
4,"(-0.155, -0.104]",55,0.527
5,"(-0.104, -0.0613]",55,0.564
6,"(-0.0613, -0.0162]",55,0.618
7,"(-0.0162, 0.0503]",55,0.509
8,"(0.0503, 0.145]",55,0.564
9,"(0.145, 0.716]",55,0.618



Validation quantile hit-rate: quality_score


...

,quantile,games,win_rate
0,"(0.477, 0.562]",510,0.547
1,"(0.562, 0.714]",41,0.585



Monotonicity summary


,metric,monotonicity_fraction,lift_last_minus_first
0,score_raw,0.667,0.100
1,quality_score,0.556,0.084
2,coherence_score,0.444,-0.045
3,p_model_logit,0.333,-0.098
4,p_model_iso,1.000,0.038



Year summary row


,season,rows_scored,rows_with_outcome,train_rows,valid_rows,top_n_default,topn_rows,topn_win_rate,best_model_by_brier,best_brier,best_logloss,best_auc
0,2025,2581,2581,2030,551,3,589,0.563667,logit,0.248032,0.689219,0.472845


## Metric definitions and acceptance bars (winner prediction)

- **Brier score**: mean squared error of predicted probability vs actual outcome (`0/1`). Lower is better.
- **Log loss**: penalizes confident wrong probabilities more than Brier. Lower is better.
- **AUC**: probability that a random win is ranked above a random loss. `0.5` is random, `1.0` is perfect.
- **Monotonicity fraction**: fraction of adjacent quantile bins where win rate moves in expected direction.
- **Lift (last-first)**: difference in win rate between highest and lowest quantile bins.
- **Top-N win rate**: win rate among the top-N ranked picks per slate date.

Suggested minimum bars for practical actionability (outright winners):

- `AUC >= 0.53`
- `best_brier <= 0.245`
- `monotonicity_fraction >= 0.60` on calibrated probability
- `lift_last_minus_first >= 0.03` on calibrated probability
- `topn_win_rate >= 0.54` for `TOP_N=3`


In [17]:
# Acceptance-bar check for actionability (winner prediction)
# Requires year_summary, metrics_df, mono_df from the calibration cell.

if "year_summary" not in globals() or "metrics_df" not in globals() or "mono_df" not in globals():
    raise ValueError("Run calibration/summary cell first.")

bars = {
    "min_auc": 0.53,
    "max_brier": 0.245,
    "min_mono": 0.60,
    "min_lift": 0.03,
    "min_topn_win_rate": 0.54,
}

ys = year_summary.iloc[0]
prob_row = mono_df[mono_df["metric"].isin(["p_model_logit", "p_model_iso"])].copy()
if prob_row.empty:
    mono_best = np.nan
    lift_best = np.nan
else:
    mono_best = float(prob_row["monotonicity_fraction"].max())
    lift_best = float(prob_row["lift_last_minus_first"].max())

checks = pd.DataFrame(
    [
        {"check": "AUC", "value": float(ys.get("best_auc", np.nan)), "bar": bars["min_auc"], "pass": float(ys.get("best_auc", np.nan)) >= bars["min_auc"]},
        {"check": "Brier", "value": float(ys.get("best_brier", np.nan)), "bar": bars["max_brier"], "pass": float(ys.get("best_brier", np.nan)) <= bars["max_brier"]},
        {"check": "Monotonicity (prob)", "value": mono_best, "bar": bars["min_mono"], "pass": mono_best >= bars["min_mono"] if pd.notna(mono_best) else False},
        {"check": "Lift (prob)", "value": lift_best, "bar": bars["min_lift"], "pass": lift_best >= bars["min_lift"] if pd.notna(lift_best) else False},
        {"check": "Top-3 win rate", "value": float(ys.get("topn_win_rate", np.nan)), "bar": bars["min_topn_win_rate"], "pass": float(ys.get("topn_win_rate", np.nan)) >= bars["min_topn_win_rate"]},
    ]
)

print("Actionability checks")
display(checks)
print("Passed", int(checks["pass"].sum()), "of", len(checks), "checks")

Actionability checks


,check,value,bar,pass
0,AUC,0.472845,0.530,False
1,Brier,0.248032,0.245,False
2,Monotonicity (prob),1.000000,0.600,True
3,Lift (prob),0.038000,0.030,True
4,Top-3 win rate,0.563667,0.540,True


Passed 3 of 5 checks
